In [1]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 974.8/974.8 kB 15.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.8 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.3 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 28.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 12.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.0 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 37.3 MB/s eta 0:00:0000:0100:01m
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.8.93
    Uninstalling nvidia-nvjitlink-cu12-12.8.93:
      Successfully uninstalled nvidia-nvjitlink-cu12-12.8.93
  Attempting uninstall: nvidia-curand-cu12
    Found

In [2]:
# Imports

import os
import numpy as np
import pandas as pd
from PIL import Image
import shutil
import time
import yaml
from pathlib import Path
from tqdm.notebook import tqdm 

import torch
import random
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from ultralytics import YOLO
import json

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
# Set seeds 

np.random.seed(42)
random.seed(42)
torch.manual_seed(42)

In [4]:
# Paths 

# BYU dataset 
data_path = "/kaggle/input/byu-locating-bacterial-flagellar-motors-2025/"
train_dir = os.path.join(data_path, "train")  
labels_csv_path = os.path.join(data_path, "train_labels.csv")

# cryoET dataset 
NEW_DATA_DIR = "/kaggle/input/cryoet-flagellar-motors-dataset"
NEW_SLICES_ROOT = os.path.join(NEW_DATA_DIR, "jpgs")
NEW_LABELS_CSV = os.path.join(NEW_DATA_DIR, "labels.csv")

# YOLO structure
yolo_dataset_dir = "/kaggle/working/yolo_dataset"
yolo_images_train = os.path.join(yolo_dataset_dir, "images", "train")
yolo_images_val = os.path.join(yolo_dataset_dir, "images", "val")
yolo_labels_train = os.path.join(yolo_dataset_dir, "labels", "train")
yolo_labels_val = os.path.join(yolo_dataset_dir, "labels", "val")
yolo_weights_dir = "/kaggle/working/yolo_weights"

for dir_path in [yolo_images_train, yolo_images_val, yolo_labels_train, yolo_labels_val, yolo_weights_dir]:
    os.makedirs(dir_path, exist_ok=True)

TRUST = 4         # Number of slices above and below the motor slice to include
BOX_SIZE = 24     # Bounding box size in pixels
TRAIN_SPLIT = 0.8 # Tomogram-level train/val split 

yolo_pretrained_weights = "/kaggle/working/yolo_weights/yolov8l.pt"

new_processed_dir = "/kaggle/working/new_processed_slices"
os.makedirs(new_processed_dir, exist_ok=True)

In [5]:
# Image Normalization Function

def normalize_slice(slice_data):
    """
    Normalize slice data using 2nd and 98th percentiles.
    """
    p2 = np.percentile(slice_data, 2)
    p98 = np.percentile(slice_data, 98)
    clipped_data = np.clip(slice_data, p2, p98)
    normalized = 255 * (clipped_data - p2) / (p98 - p2)
    return np.uint8(normalized)

In [ ]:
# Preprocess cryoET Dataset Slices

def preprocess_new_slices(target_size=(128, 512, 512)):
    """
    Process each slice from the new dataset:
      - For each tomogram folder in NEW_SLICES_ROOT, iterate over slice images.
      - For each slice: open, normalize, resize to target_size, and save into new_processed_dir.
      - The saved filename will be: {tomo_id}_slice_{slice_index:04d}.jpg
    """

    tomo_folders = [f for f in os.listdir(NEW_SLICES_ROOT) if os.path.isdir(os.path.join(NEW_SLICES_ROOT, f))]
    for tomo_id in tqdm(tomo_folders, desc="Preprocessing new slices"):
        tomo_path = os.path.join(NEW_SLICES_ROOT, tomo_id)
        slice_files = sorted([f for f in os.listdir(tomo_path) if f.lower().endswith(('.png','.jpg','.jpeg','.tif','.tiff'))])
        for sf in slice_files:
            slice_path = os.path.join(tomo_path, sf)
            try:
                img = Image.open(slice_path)
            except Exception as e:
                print(f"Error opening {slice_path}: {e}")
                continue
            img_array = np.array(img)
            norm_img = normalize_slice(img_array)
            resized_img = Image.fromarray(norm_img).resize((target_size[2], target_size[1]))  
            try:
                base = os.path.splitext(sf)[0]
                parts = base.split('_')
                slice_index = int(parts[-1])
            except:
                slice_index = 0
            out_filename = f"{tomo_id}_slice_{slice_index:04d}.jpg"
            out_path = os.path.join(new_processed_dir, out_filename)
            resized_img.save(out_path)
    print("cryoET dataset slices have been preprocessed.")

preprocess_new_slices()

Preprocessing new slices:   0%|          | 0/1288 [00:00<?, ?it/s]

In [ ]:
# Convert cryoET Dataset to YOLO Format and Merge

def process_new_dataset_to_yolo():
    """
    Process the new dataset by reading its CSV (using columns: tomo_id, x, y, z, dataset),
    dropping the 'dataset' column, and converting each annotation to YOLO format.
    For each row, it locates the preprocessed slice image from new_processed_dir corresponding
    to the slice index (z), copies it into the YOLO training images folder, and writes a YOLO label
    file with normalized coordinates.
    """
    df_new = pd.read_csv(NEW_LABELS_CSV)
    print(f"CryoET dataset CSV loaded with {len(df_new)} entries.")
    
    for idx, row in tqdm(df_new.iterrows(), total=len(df_new), desc="Processing new dataset annotations"):
        tomo_id = row["tomo_id"].strip()
        motor_slice = int(round(row["z"]))  
        motor_x = int(round(row["x"]))
        motor_y = int(round(row["y"]))
        
        src_filename = f"{tomo_id}_slice_{motor_slice:04d}.jpg"
        src_path = os.path.join(new_processed_dir, src_filename)
        if not os.path.exists(src_path):
            print(f"Warning: {src_path} not found. Skipping.")
            continue
        dest_path = os.path.join(yolo_images_train, src_filename)
        shutil.copy(src_path, dest_path)
        
        img_width, img_height = 512, 512
        x_center_norm = motor_x / img_width
        y_center_norm = motor_y / img_height
        box_width_norm = BOX_SIZE / img_width
        box_height_norm = BOX_SIZE / img_height
        
        label_filename = src_filename.replace('.jpg', '.txt')
        label_path = os.path.join(yolo_labels_train, label_filename)
        with open(label_path, "w") as f:
            f.write(f"0 {x_center_norm:.6f} {y_center_norm:.6f} {box_width_norm:.6f} {box_height_norm:.6f}\n")
    print("cryoET dataset converted to YOLO format and merged.")

process_new_dataset_to_yolo()

In [ ]:
# YOLO YAML Configuration

def prepare_dataset():
    yaml_out_path = os.path.join(yolo_dataset_dir, 'dataset.yaml')
    yaml_content = {
        'path': yolo_dataset_dir,
        'train': 'images/train',
        'val': 'images/val', 
        'names': {0: 'motor'}
    }
    with open(yaml_out_path, 'w') as f:
        yaml.dump(yaml_content, f, default_flow_style=False)
    print(f"YAML configuration saved at: {yaml_out_path}")
    return yaml_out_path

yaml_path = prepare_dataset()

def fix_yaml_paths(yaml_path):
    with open(yaml_path, 'r') as f:
        yaml_data = yaml.safe_load(f)
    if 'path' in yaml_data:
        yaml_data['path'] = yolo_dataset_dir
    fixed_yaml_path = os.path.join("/kaggle/working", "fixed_dataset.yaml")
    with open(fixed_yaml_path, 'w') as f:
        yaml.dump(yaml_data, f)
    print(f"Fixed YAML saved at {fixed_yaml_path}.")
    return fixed_yaml_path

In [ ]:
def plot_dfl_loss_curve(run_dir):
    results_csv = os.path.join(run_dir, 'results.csv')
    if not os.path.exists(results_csv):
        print(f"Results CSV not found at {results_csv}")
        return
    results_df = pd.read_csv(results_csv)
    train_dfl_col = [col for col in results_df.columns if 'train/dfl_loss' in col]
    val_dfl_col = [col for col in results_df.columns if 'val/dfl_loss' in col]
    if not train_dfl_col or not val_dfl_col:
        print("DFL loss columns not found. Available columns:", results_df.columns.tolist())
        return
    train_dfl_col = train_dfl_col[0]
    val_dfl_col = val_dfl_col[0]
    best_epoch = results_df[val_dfl_col].idxmin()
    best_val_loss = results_df.loc[best_epoch, val_dfl_col]
    plt.figure(figsize=(10,6))
    plt.plot(results_df['epoch'], results_df[train_dfl_col], label='Train DFL Loss')
    plt.plot(results_df['epoch'], results_df[val_dfl_col], label='Validation DFL Loss')
    plt.axvline(x=results_df.loc[best_epoch, 'epoch'], color='r', linestyle='--',
                label=f'Best Model (Epoch {int(results_df.loc[best_epoch, "epoch"])}, Loss: {best_val_loss:.4f})')
    plt.xlabel("Epoch")
    plt.ylabel("DFL Loss")
    plt.title("Training vs Validation Loss")
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)
    plot_path = os.path.join(run_dir, 'dfl_loss_curve.png')
    plt.savefig(plot_path)
    plt.savefig(os.path.join("/kaggle/working", "dfl_loss_curve.png"))
    print(f"Loss curve saved to {plot_path}")
    plt.close()
    return best_epoch, best_val_loss

In [ ]:
# YOLO Training and Prediction Helper Functions

def train_yolo_model(yaml_path, pretrained_weights_path, epochs=100, batch_size=64, img_size=640):
    print(f"Loading pretrained weights from {pretrained_weights_path}")
    model = YOLO(pretrained_weights_path)
    results = model.train(
        data=yaml_path,
        epochs=epochs,
        batch=batch_size,
        imgsz=img_size,
        project=yolo_weights_dir,
        name='motor_detector',
        exist_ok=True,
        patience=5,
        save_period=5,
        val=True,
        verbose=True
    )
    run_dir = os.path.join(yolo_weights_dir, 'motor_detector')
    best_epoch_info = plot_dfl_loss_curve(run_dir)
    if best_epoch_info:
        best_epoch, best_val_loss = best_epoch_info
        print(f"Best model: Epoch {best_epoch}, Val Loss: {best_val_loss:.4f}")
    return model, results

def predict_on_samples(model, num_samples=4):
    val_dir = os.path.join(yolo_dataset_dir, 'images', 'val')
    if not os.path.exists(val_dir) or len(os.listdir(val_dir)) == 0:
        print("No validation images found; using training images for prediction.")
        val_dir = os.path.join(yolo_dataset_dir, 'images', 'train')
    val_images = os.listdir(val_dir)
    samples = np.random.choice(val_images, min(num_samples, len(val_images)), replace=False)
    fig, axes = plt.subplots(2, 2, figsize=(12, 12))
    axes = axes.flatten()
    for i, img_file in enumerate(samples):
        img_path = os.path.join(val_dir, img_file)
        results = model.predict(img_path, conf=0.25)[0]
        img = Image.open(img_path)
        axes[i].imshow(np.array(img))
        axes[i].set_title(f"{img_file}")
    plt.tight_layout()
    plt.savefig(os.path.join("/kaggle/working", "predictions.png"))
    plt.show()

print("YOLO helper functions defined.")

In [ ]:
# Train YOLO 

def main():
    print("Starting YOLO training")
    yaml_path = prepare_dataset()
    print(f"Using YAML: {yaml_path}")
    with open(yaml_path, 'r') as f:
        print("YAML contents:\n", f.read())
    print("Training YOLO model...")
    model, results = train_yolo_model(
        yaml_path,
        pretrained_weights_path=yolo_pretrained_weights,
        epochs=100,
        batch_size=16,
        img_size=640
    )
    print("Training complete!")
    print("Running sample predictions")
    predict_on_samples(model, num_samples=4)

In [ ]:
if __name__ == "__main__":
    main()